# Taller: ABC para resolver un VRP usando `vrplib` + OOP

**Curso:** Computación Blanda  
**Tema:** Artificial Bee Colony (ABC) aplicado al Capacitated Vehicle Routing Problem (CVRP)  
**Modalidad:** Taller experimental en Jupyter Notebook  
**Enfoque:** ajuste de hiperparámetros, calidad de solución, tiempo de ejecución y comparación con **ACO** y **CBGA**

---

## Propósito

En este taller el estudiante implementará un solucionador del problema de ruteo de vehículos con capacidad (**CVRP**) utilizando la metaheurística **Artificial Bee Colony (ABC)**, organizada con un diseño **OOP + `dataclasses`**, cargando instancias desde archivos **VRPLIB** mediante la librería `vrplib`.

El trabajo culmina con una **comparación experimental** entre tres enfoques:

1. **ABC** (Artificial Bee Colony)  
2. **ACO** (Ant Colony Optimization)  
3. **CBGA** (Chu-Beasley Genetic Algorithm, versión simplificada para taller)

---

## Objetivos de aprendizaje

Al finalizar este taller, el estudiante podrá:

- modelar un CVRP a partir de una instancia VRPLIB;
- representar una solución como un conjunto de rutas factibles;
- implementar una versión funcional de ABC para VRP;
- identificar el efecto de los hiperparámetros sobre la calidad y el tiempo;
- comparar experimentalmente ABC contra ACO y CBGA;
- argumentar, con métricas, cuál enfoque ofrece mejor compromiso entre **calidad** y **costo computacional**.

---

## Competencias

- Modelado de problemas combinatorios.
- Diseño orientado a objetos para metaheurísticas.
- Experimentación reproducible.
- Análisis cuantitativo de resultados.
- Interpretación crítica de hiperparámetros.

## Problema: CVRP

En el **Capacitated Vehicle Routing Problem** se tiene:

- un depósito;
- un conjunto de clientes con demanda;
- una flota de vehículos con capacidad limitada.

El objetivo es encontrar rutas que:

1. inicien y terminen en el depósito,
2. visiten todos los clientes exactamente una vez,
3. respeten la capacidad del vehículo,
4. minimicen la distancia total recorrida.

---

## Idea de ABC en este contexto

En ABC hay tres tipos de abejas:

- **Employed bees**: exploran variaciones de soluciones actuales.
- **Onlooker bees**: eligen soluciones prometedoras y las refinan.
- **Scout bees**: reemplazan fuentes que se estancaron.

En VRP, una **fuente de alimento** representa una **solución completa del CVRP**, es decir, un conjunto de rutas factibles.

In [ ]:

# Instalación sugerida (ejecute solo si hace falta)
# %pip install vrplib matplotlib pandas numpy

In [ ]:

from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
import math
import random
import time
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import vrplib
except ImportError:
    vrplib = None
    print("Advertencia: vrplib no está instalado. Instálelo con: pip install vrplib")

## 1. Modelo de datos

Usaremos `dataclasses` para que la representación sea clara, compacta y fácil de depurar.

In [ ]:

@dataclass
class VRPInstance:
    name: str
    depot: int
    capacity: int
    coords: Dict[int, Tuple[float, float]]
    demands: Dict[int, int]
    customers: List[int]
    distance: np.ndarray

    @property
    def n_customers(self) -> int:
        return len(self.customers)


@dataclass
class VRPSolution:
    routes: List[List[int]]
    cost: float = math.inf
    loads: List[int] = field(default_factory=list)

    def copy(self) -> "VRPSolution":
        return VRPSolution(
            routes=[r[:] for r in self.routes],
            cost=self.cost,
            loads=self.loads[:],
        )

In [ ]:

def euclidean(a: Tuple[float, float], b: Tuple[float, float]) -> float:
    return math.dist(a, b)


def build_distance_matrix(coords: Dict[int, Tuple[float, float]]) -> np.ndarray:
    nodes = sorted(coords.keys())
    max_id = max(nodes)
    d = np.zeros((max_id + 1, max_id + 1), dtype=float)
    for i in nodes:
        for j in nodes:
            if i != j:
                d[i, j] = euclidean(coords[i], coords[j])
    return d


def load_vrplib_instance(path: str) -> VRPInstance:
    if vrplib is None:
        raise ImportError("Debe instalar vrplib antes de cargar instancias.")
    data = vrplib.read_instance(path)

    name = data.get("name", "VRP")
    capacity = int(data["capacity"])

    coord_section = data["node_coord"]
    demand_section = data["demand"]

    coords = {int(k): tuple(map(float, v)) for k, v in coord_section.items()}
    demands = {int(k): int(v) for k, v in demand_section.items()}

    depot_candidates = data.get("depot", [1])
    if isinstance(depot_candidates, (list, tuple)):
        depot = int(depot_candidates[0])
    else:
        depot = int(depot_candidates)

    customers = [n for n in sorted(coords.keys()) if n != depot]
    distance = build_distance_matrix(coords)

    return VRPInstance(
        name=name,
        depot=depot,
        capacity=capacity,
        coords=coords,
        demands=demands,
        customers=customers,
        distance=distance,
    )

## 2. Funciones de evaluación

La calidad de una solución depende de:

- distancia total;
- factibilidad respecto a capacidad.

En este taller usaremos soluciones **factibles** desde la construcción.  
Más adelante puede añadirse penalización para permitir soluciones temporales inviables.

In [ ]:

def route_load(inst: VRPInstance, route: List[int]) -> int:
    return sum(inst.demands[c] for c in route)


def route_cost(inst: VRPInstance, route: List[int]) -> float:
    if not route:
        return 0.0
    depot = inst.depot
    total = inst.distance[depot, route[0]]
    for i in range(len(route) - 1):
        total += inst.distance[route[i], route[i + 1]]
    total += inst.distance[route[-1], depot]
    return total


def evaluate_solution(inst: VRPInstance, sol: VRPSolution) -> VRPSolution:
    loads = [route_load(inst, r) for r in sol.routes]
    cost = sum(route_cost(inst, r) for r in sol.routes)
    sol.loads = loads
    sol.cost = cost
    return sol


def is_feasible(inst: VRPInstance, sol: VRPSolution) -> bool:
    seen = []
    for route in sol.routes:
        if route_load(inst, route) > inst.capacity:
            return False
        seen.extend(route)
    return sorted(seen) == sorted(inst.customers)

## 3. Constructores de solución

Generaremos soluciones iniciales con una heurística aleatoria factible tipo **split secuencial**.

In [ ]:

def random_greedy_solution(inst: VRPInstance, rng: random.Random) -> VRPSolution:
    customers = inst.customers[:]
    rng.shuffle(customers)

    routes = []
    current = []
    load = 0

    for c in customers:
        d = inst.demands[c]
        if load + d <= inst.capacity:
            current.append(c)
            load += d
        else:
            routes.append(current)
            current = [c]
            load = d

    if current:
        routes.append(current)

    sol = VRPSolution(routes=routes)
    return evaluate_solution(inst, sol)


def nearest_neighbor_seed(inst: VRPInstance, rng: random.Random) -> VRPSolution:
    remaining = set(inst.customers)
    routes = []

    while remaining:
        route = []
        load = 0
        current = inst.depot
        while True:
            feasible = [c for c in remaining if load + inst.demands[c] <= inst.capacity]
            if not feasible:
                break
            nxt = min(feasible, key=lambda c: inst.distance[current, c])
            route.append(nxt)
            remaining.remove(nxt)
            load += inst.demands[nxt]
            current = nxt
        routes.append(route)

    sol = VRPSolution(routes=routes)
    return evaluate_solution(inst, sol)

## 4. Operadores de vecindad para VRP

Usaremos operadores simples y efectivos:

- **swap intra/inter-ruta**
- **relocate**
- **2-opt intra-ruta**

Estos operadores serán reutilizados por ABC, ACO y CBGA.

In [ ]:

def try_2opt_route(inst: VRPInstance, route: List[int], rng: random.Random) -> List[int]:
    if len(route) < 4:
        return route[:]
    i, j = sorted(rng.sample(range(len(route)), 2))
    if i == j:
        return route[:]
    new_route = route[:i] + route[i:j+1][::-1] + route[j+1:]
    return new_route


def relocate_move(inst: VRPInstance, sol: VRPSolution, rng: random.Random) -> VRPSolution:
    sol = sol.copy()
    non_empty = [idx for idx, r in enumerate(sol.routes) if r]
    if not non_empty:
        return sol

    r1 = rng.choice(non_empty)
    cidx = rng.randrange(len(sol.routes[r1]))
    customer = sol.routes[r1].pop(cidx)

    target_routes = list(range(len(sol.routes)))
    rng.shuffle(target_routes)

    inserted = False
    for r2 in target_routes:
        new_load = route_load(inst, sol.routes[r2]) + inst.demands[customer]
        if new_load <= inst.capacity:
            pos = rng.randrange(len(sol.routes[r2]) + 1)
            sol.routes[r2].insert(pos, customer)
            inserted = True
            break

    if not inserted:
        sol.routes[r1].insert(min(cidx, len(sol.routes[r1])), customer)

    sol.routes = [r for r in sol.routes if r]
    return evaluate_solution(inst, sol)


def swap_move(inst: VRPInstance, sol: VRPSolution, rng: random.Random) -> VRPSolution:
    sol = sol.copy()
    non_empty = [i for i, r in enumerate(sol.routes) if r]
    if len(non_empty) == 0:
        return sol

    r1 = rng.choice(non_empty)
    r2 = rng.choice(non_empty)
    i = rng.randrange(len(sol.routes[r1]))
    j = rng.randrange(len(sol.routes[r2]))

    a = sol.routes[r1][i]
    b = sol.routes[r2][j]

    if r1 != r2:
        load1 = route_load(inst, sol.routes[r1]) - inst.demands[a] + inst.demands[b]
        load2 = route_load(inst, sol.routes[r2]) - inst.demands[b] + inst.demands[a]
        if load1 > inst.capacity or load2 > inst.capacity:
            return evaluate_solution(inst, sol)

    sol.routes[r1][i], sol.routes[r2][j] = sol.routes[r2][j], sol.routes[r1][i]
    return evaluate_solution(inst, sol)


def two_opt_move(inst: VRPInstance, sol: VRPSolution, rng: random.Random) -> VRPSolution:
    sol = sol.copy()
    candidates = [i for i, r in enumerate(sol.routes) if len(r) >= 4]
    if not candidates:
        return sol
    r = rng.choice(candidates)
    sol.routes[r] = try_2opt_route(inst, sol.routes[r], rng)
    return evaluate_solution(inst, sol)


def perturb(inst: VRPInstance, sol: VRPSolution, rng: random.Random) -> VRPSolution:
    move = rng.choice([relocate_move, swap_move, two_opt_move])
    return move(inst, sol, rng)

## 5. ABC para VRP

### Hiperparámetros principales

- `colony_size`: tamaño total de la colonia.
- `limit`: número de intentos sin mejora antes de convertir una fuente en scout.
- `max_iters`: iteraciones máximas.
- `neighborhood_trials`: cuántos vecinos se prueban por exploración.
- `seed_mode`: estrategia para generar soluciones iniciales.
- `elite_local_search`: refinamiento adicional sobre la mejor solución.

### Pregunta guía

> ¿Cuál de estos hiperparámetros impacta más la **calidad** y cuál impacta más el **tiempo**?

In [ ]:

@dataclass
class ABCConfig:
    colony_size: int = 30
    limit: int = 30
    max_iters: int = 200
    neighborhood_trials: int = 3
    seed: int = 7
    seed_mode: str = "mixed"   # random | nn | mixed
    elite_local_search: bool = True


@dataclass
class FoodSource:
    solution: VRPSolution
    trials: int = 0


class ABCVRPSolver:
    def __init__(self, config: ABCConfig):
        self.config = config
        self.rng = random.Random(config.seed)

    def _make_solution(self, inst: VRPInstance) -> VRPSolution:
        mode = self.config.seed_mode
        if mode == "random":
            return random_greedy_solution(inst, self.rng)
        if mode == "nn":
            return nearest_neighbor_seed(inst, self.rng)
        return self.rng.choice([
            lambda: random_greedy_solution(inst, self.rng),
            lambda: nearest_neighbor_seed(inst, self.rng),
        ])()

    def _init_sources(self, inst: VRPInstance) -> List[FoodSource]:
        n_sources = max(2, self.config.colony_size // 2)
        return [FoodSource(self._make_solution(inst)) for _ in range(n_sources)]

    def _neighbor(self, inst: VRPInstance, sol: VRPSolution) -> VRPSolution:
        best = sol.copy()
        for _ in range(self.config.neighborhood_trials):
            cand = perturb(inst, sol, self.rng)
            if cand.cost < best.cost:
                best = cand
        return best

    def _fitness(self, cost: float) -> float:
        return 1.0 / (1.0 + cost)

    def solve(self, inst: VRPInstance) -> dict:
        start = time.perf_counter()
        sources = self._init_sources(inst)
        best = min((fs.solution for fs in sources), key=lambda s: s.cost).copy()
        history = []

        for _ in range(self.config.max_iters):
            # Employed bees
            for fs in sources:
                cand = self._neighbor(inst, fs.solution)
                if cand.cost < fs.solution.cost:
                    fs.solution = cand
                    fs.trials = 0
                else:
                    fs.trials += 1

            # Onlookers
            fits = np.array([self._fitness(fs.solution.cost) for fs in sources], dtype=float)
            probs = fits / fits.sum()
            for _ in range(len(sources)):
                idx = np.random.choice(len(sources), p=probs)
                fs = sources[idx]
                cand = self._neighbor(inst, fs.solution)
                if cand.cost < fs.solution.cost:
                    fs.solution = cand
                    fs.trials = 0
                else:
                    fs.trials += 1

            # Scouts
            for fs in sources:
                if fs.trials >= self.config.limit:
                    fs.solution = self._make_solution(inst)
                    fs.trials = 0

            current_best = min((fs.solution for fs in sources), key=lambda s: s.cost)
            if current_best.cost < best.cost:
                best = current_best.copy()
                if self.config.elite_local_search:
                    for _ in range(5):
                        improved = self._neighbor(inst, best)
                        if improved.cost < best.cost:
                            best = improved

            history.append(best.cost)

        elapsed = time.perf_counter() - start
        return {
            "best_solution": best,
            "best_cost": best.cost,
            "time_sec": elapsed,
            "history": history,
            "feasible": is_feasible(inst, best),
            "config": self.config,
        }

## 6. Baselines de comparación: ACO y CBGA

Aquí se incluyen implementaciones didácticas y compactas, suficientes para comparar:

- comportamiento general,
- calidad de la solución,
- sensibilidad a hiperparámetros,
- tiempo de ejecución.

No pretenden ser versiones de competencia, sino **baselines de curso**.

In [ ]:

@dataclass
class ACOConfig:
    n_ants: int = 20
    alpha: float = 1.0
    beta: float = 3.0
    evaporation: float = 0.3
    q: float = 100.0
    max_iters: int = 120
    seed: int = 11


class ACOVRPSolver:
    def __init__(self, config: ACOConfig):
        self.config = config
        self.rng = random.Random(config.seed)

    def _build_solution(self, inst: VRPInstance, tau: np.ndarray) -> VRPSolution:
        remaining = set(inst.customers)
        routes = []
        depot = inst.depot

        while remaining:
            route = []
            current = depot
            load = 0

            while True:
                feasible = [c for c in remaining if load + inst.demands[c] <= inst.capacity]
                if not feasible:
                    break

                desirabilities = []
                for c in feasible:
                    pher = tau[current, c] ** self.config.alpha
                    heur = (1.0 / (inst.distance[current, c] + 1e-9)) ** self.config.beta
                    desirabilities.append(pher * heur)

                total = sum(desirabilities)
                probs = [d / total for d in desirabilities]
                chosen = self.rng.choices(feasible, weights=probs, k=1)[0]

                route.append(chosen)
                remaining.remove(chosen)
                load += inst.demands[chosen]
                current = chosen

            routes.append(route)

        sol = VRPSolution(routes=routes)
        return evaluate_solution(inst, sol)

    def solve(self, inst: VRPInstance) -> dict:
        start = time.perf_counter()
        n = max(inst.coords.keys()) + 1
        tau = np.ones((n, n), dtype=float)
        best = None
        history = []

        for _ in range(self.config.max_iters):
            ants = [self._build_solution(inst, tau) for _ in range(self.config.n_ants)]
            iteration_best = min(ants, key=lambda s: s.cost)
            if best is None or iteration_best.cost < best.cost:
                best = iteration_best.copy()

            tau *= (1.0 - self.config.evaporation)

            for sol in ants:
                deposit = self.config.q / sol.cost
                for route in sol.routes:
                    prev = inst.depot
                    for c in route:
                        tau[prev, c] += deposit
                        tau[c, prev] += deposit
                        prev = c
                    tau[prev, inst.depot] += deposit
                    tau[inst.depot, prev] += deposit

            history.append(best.cost)

        elapsed = time.perf_counter() - start
        return {
            "best_solution": best,
            "best_cost": best.cost,
            "time_sec": elapsed,
            "history": history,
            "feasible": is_feasible(inst, best),
            "config": self.config,
        }

In [ ]:

@dataclass
class CBGAConfig:
    population_size: int = 40
    generations: int = 160
    crossover_rate: float = 0.9
    mutation_rate: float = 0.25
    tournament_k: int = 3
    seed: int = 19


class CBGAVRPSolver:
    def __init__(self, config: CBGAConfig):
        self.config = config
        self.rng = random.Random(config.seed)

    def _chromosome_to_solution(self, inst: VRPInstance, chrom: List[int]) -> VRPSolution:
        routes = []
        current = []
        load = 0
        for c in chrom:
            d = inst.demands[c]
            if load + d <= inst.capacity:
                current.append(c)
                load += d
            else:
                routes.append(current)
                current = [c]
                load = d
        if current:
            routes.append(current)
        return evaluate_solution(inst, VRPSolution(routes=routes))

    def _random_chromosome(self, inst: VRPInstance) -> List[int]:
        chrom = inst.customers[:]
        self.rng.shuffle(chrom)
        return chrom

    def _fitness(self, inst: VRPInstance, chrom: List[int]) -> float:
        return self._chromosome_to_solution(inst, chrom).cost

    def _tournament(self, inst: VRPInstance, pop: List[List[int]]) -> List[int]:
        sample = self.rng.sample(pop, self.config.tournament_k)
        return min(sample, key=lambda ch: self._fitness(inst, ch))

    def _ox(self, p1: List[int], p2: List[int]) -> List[int]:
        n = len(p1)
        a, b = sorted(self.rng.sample(range(n), 2))
        child = [None] * n
        child[a:b+1] = p1[a:b+1]
        fill = [g for g in p2 if g not in child]
        ptr = 0
        for i in range(n):
            if child[i] is None:
                child[i] = fill[ptr]
                ptr += 1
        return child

    def _mutate(self, chrom: List[int]) -> None:
        if self.rng.random() < self.config.mutation_rate:
            i, j = sorted(self.rng.sample(range(len(chrom)), 2))
            chrom[i], chrom[j] = chrom[j], chrom[i]

    def solve(self, inst: VRPInstance) -> dict:
        start = time.perf_counter()
        pop = [self._random_chromosome(inst) for _ in range(self.config.population_size)]
        best_sol = None
        history = []

        for _ in range(self.config.generations):
            new_pop = []

            elite = min(pop, key=lambda ch: self._fitness(inst, ch))
            new_pop.append(elite[:])

            while len(new_pop) < self.config.population_size:
                p1 = self._tournament(inst, pop)
                p2 = self._tournament(inst, pop)
                if self.rng.random() < self.config.crossover_rate:
                    child = self._ox(p1, p2)
                else:
                    child = p1[:]
                self._mutate(child)
                new_pop.append(child)

            pop = new_pop
            gen_best = min(pop, key=lambda ch: self._fitness(inst, ch))
            sol = self._chromosome_to_solution(inst, gen_best)
            if best_sol is None or sol.cost < best_sol.cost:
                best_sol = sol.copy()
            history.append(best_sol.cost)

        elapsed = time.perf_counter() - start
        return {
            "best_solution": best_sol,
            "best_cost": best_sol.cost,
            "time_sec": elapsed,
            "history": history,
            "feasible": is_feasible(inst, best_sol),
            "config": self.config,
        }

## 7. Visualización

Estas funciones ayudan a inspeccionar rutas y curvas de convergencia.

In [ ]:

def plot_solution(inst: VRPInstance, sol: VRPSolution, title: str = "Solución VRP") -> None:
    depot_xy = inst.coords[inst.depot]
    plt.figure(figsize=(8, 6))
    plt.scatter([depot_xy[0]], [depot_xy[1]], s=160, marker="s", label="Depósito")

    for c in inst.customers:
        x, y = inst.coords[c]
        plt.scatter([x], [y], s=40)
        plt.text(x, y, str(c), fontsize=8)

    for route in sol.routes:
        path = [inst.depot] + route + [inst.depot]
        xs = [inst.coords[n][0] for n in path]
        ys = [inst.coords[n][1] for n in path]
        plt.plot(xs, ys)

    plt.title(f"{title} | costo={sol.cost:.2f}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_histories(results: Dict[str, dict]) -> None:
    plt.figure(figsize=(9, 5))
    for name, res in results.items():
        plt.plot(res["history"], label=name)
    plt.xlabel("Iteración / generación")
    plt.ylabel("Mejor costo acumulado")
    plt.title("Convergencia")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 8. Cargar una instancia VRPLIB

Use una instancia como:

- `A-n32-k5.vrp`
- `A-n45-k6.vrp`
- `A-n60-k9.vrp`

Ajuste la ruta del archivo en la siguiente celda.

In [ ]:

# Ejemplo:
# inst = load_vrplib_instance("A-n32-k5.vrp")
# print(inst)

## 9. Ejecutar ABC, ACO y CBGA

En este bloque se propone una configuración inicial razonable.  
La idea del taller no es quedarse con esta configuración, sino **experimentar**.

In [ ]:

# Descomente cuando tenga la instancia cargada:
#
# abc = ABCVRPSolver(ABCConfig(
#     colony_size=30,
#     limit=25,
#     max_iters=150,
#     neighborhood_trials=4,
#     seed=1,
#     seed_mode="mixed",
#     elite_local_search=True,
# ))
#
# aco = ACOVRPSolver(ACOConfig(
#     n_ants=20,
#     alpha=1.0,
#     beta=3.0,
#     evaporation=0.25,
#     q=100.0,
#     max_iters=120,
#     seed=2,
# ))
#
# cbga = CBGAVRPSolver(CBGAConfig(
#     population_size=40,
#     generations=140,
#     crossover_rate=0.9,
#     mutation_rate=0.2,
#     tournament_k=3,
#     seed=3,
# ))
#
# res_abc = abc.solve(inst)
# res_aco = aco.solve(inst)
# res_cbga = cbga.solve(inst)
#
# results = {
#     "ABC": res_abc,
#     "ACO": res_aco,
#     "CBGA": res_cbga,
# }
#
# summary = pd.DataFrame([
#     {
#         "algoritmo": name,
#         "costo": res["best_cost"],
#         "tiempo_s": res["time_sec"],
#         "factible": res["feasible"],
#         "rutas": len(res["best_solution"].routes),
#     }
#     for name, res in results.items()
# ]).sort_values("costo")
#
# summary

In [ ]:

# Visualización sugerida:
#
# plot_histories(results)
# plot_solution(inst, res_abc["best_solution"], "ABC")
# plot_solution(inst, res_aco["best_solution"], "ACO")
# plot_solution(inst, res_cbga["best_solution"], "CBGA")

## 10. Experimento principal: ajuste de hiperparámetros de ABC

El corazón del taller está aquí.  
Se evaluará la sensibilidad de ABC respecto a sus hiperparámetros.

### Variables sugeridas

- `colony_size`: 10, 20, 30, 50
- `limit`: 10, 20, 40, 80
- `neighborhood_trials`: 1, 3, 5, 8
- `seed_mode`: `random`, `nn`, `mixed`
- `elite_local_search`: `False`, `True`

### Métricas

- mejor costo;
- tiempo total;
- desviación estándar del costo en varias corridas;
- costo promedio;
- mejor costo por segundo.

In [ ]:

def run_abc_grid(inst: VRPInstance, configs: List[ABCConfig], repeats: int = 3) -> pd.DataFrame:
    rows = []
    for cfg in configs:
        costs = []
        times = []
        feas = []

        for rep in range(repeats):
            cfg_rep = deepcopy(cfg)
            cfg_rep.seed = cfg.seed + rep
            solver = ABCVRPSolver(cfg_rep)
            res = solver.solve(inst)
            costs.append(res["best_cost"])
            times.append(res["time_sec"])
            feas.append(res["feasible"])

        rows.append({
            "colony_size": cfg.colony_size,
            "limit": cfg.limit,
            "max_iters": cfg.max_iters,
            "neighborhood_trials": cfg.neighborhood_trials,
            "seed_mode": cfg.seed_mode,
            "elite_local_search": cfg.elite_local_search,
            "mean_cost": float(np.mean(costs)),
            "std_cost": float(np.std(costs)),
            "best_cost": float(np.min(costs)),
            "mean_time": float(np.mean(times)),
            "all_feasible": all(feas),
        })

    return pd.DataFrame(rows).sort_values(["best_cost", "mean_time"])

In [ ]:

# Ejemplo de rejilla experimental:
#
# grid = [
#     ABCConfig(colony_size=20, limit=20, max_iters=120, neighborhood_trials=2, seed=10, seed_mode="random", elite_local_search=False),
#     ABCConfig(colony_size=20, limit=20, max_iters=120, neighborhood_trials=4, seed=10, seed_mode="mixed", elite_local_search=False),
#     ABCConfig(colony_size=30, limit=30, max_iters=150, neighborhood_trials=4, seed=10, seed_mode="mixed", elite_local_search=True),
#     ABCConfig(colony_size=50, limit=40, max_iters=150, neighborhood_trials=5, seed=10, seed_mode="nn", elite_local_search=True),
# ]
#
# df_grid = run_abc_grid(inst, grid, repeats=5)
# df_grid

## 11. Comparación formal contra ACO y CBGA

Una vez elegido un buen conjunto de hiperparámetros para ABC, compare contra ACO y CBGA usando:

- la misma instancia;
- varias semillas;
- mismo presupuesto aproximado de cómputo.

### Recomendación metodológica

- Haga al menos **5 corridas por algoritmo**.
- Reporte:
  - mejor costo,
  - costo promedio,
  - desviación estándar,
  - tiempo promedio.

### Discusión esperada

- ¿ABC converge rápido pero se estanca?
- ¿ACO encuentra mejores rutas cuando la heurística de distancia domina?
- ¿CBGA explora mejor el espacio pero tarda más?

In [ ]:

def benchmark_algorithms(inst: VRPInstance, repeats: int = 5) -> pd.DataFrame:
    rows = []

    for rep in range(repeats):
        abc = ABCVRPSolver(ABCConfig(
            colony_size=30, limit=30, max_iters=150,
            neighborhood_trials=4, seed=100 + rep,
            seed_mode="mixed", elite_local_search=True
        ))
        aco = ACOVRPSolver(ACOConfig(
            n_ants=20, alpha=1.0, beta=3.0,
            evaporation=0.25, q=100.0,
            max_iters=120, seed=200 + rep
        ))
        cbga = CBGAVRPSolver(CBGAConfig(
            population_size=40, generations=140,
            crossover_rate=0.9, mutation_rate=0.2,
            tournament_k=3, seed=300 + rep
        ))

        for name, solver in [("ABC", abc), ("ACO", aco), ("CBGA", cbga)]:
            res = solver.solve(inst)
            rows.append({
                "run": rep,
                "algoritmo": name,
                "costo": res["best_cost"],
                "tiempo_s": res["time_sec"],
                "factible": res["feasible"],
                "rutas": len(res["best_solution"].routes),
            })

    return pd.DataFrame(rows)

In [ ]:

# Ejemplo:
#
# bench = benchmark_algorithms(inst, repeats=5)
# bench
# bench.groupby("algoritmo")[["costo", "tiempo_s"]].agg(["mean", "std", "min"])

## 12. Actividades del estudiante

### Parte A. Comprensión del problema
1. Explique con sus palabras qué representa una solución del CVRP.
2. Justifique por qué una permutación simple de clientes no basta por sí sola para representar directamente las rutas.
3. Explique qué papel juega la capacidad del vehículo.

### Parte B. Implementación
4. Complete o mejore los operadores de vecindad.
5. Añada un operador nuevo, por ejemplo:
   - intercambio de subsecuencias,
   - destroy-and-repair,
   - cross-exchange.
6. Verifique que todas las soluciones construidas sean factibles.

### Parte C. Afinamiento de ABC
7. Ejecute al menos 12 configuraciones distintas de ABC.
8. Determine cuál hiperparámetro afecta más la calidad.
9. Determine cuál hiperparámetro afecta más el tiempo.
10. Analice si `elite_local_search` vale el costo adicional.

### Parte D. Comparación
11. Compare ABC con ACO y CBGA en una misma instancia.
12. Repita la comparación en una instancia mayor.
13. Indique en qué escenarios cada algoritmo parece más conveniente.

### Parte E. Reflexión
14. ¿Qué desventajas tiene ABC frente a ACO?
15. ¿Qué ventajas tiene ABC frente a CBGA?
16. ¿Qué cambiaría usted para llevar esta solución a una instancia de cientos de clientes?

## 13. Preguntas orientadoras para el informe

El informe del taller debe responder, como mínimo:

1. ¿Cómo se modeló el CVRP en memoria?
2. ¿Cómo representa ABC una fuente de alimento?
3. ¿Qué operadores de vecindad se usaron y por qué?
4. ¿Cuál fue la mejor configuración de ABC?
5. ¿Cuál algoritmo obtuvo el mejor costo?
6. ¿Cuál mostró mejor equilibrio entre costo y tiempo?
7. ¿Los resultados fueron estables entre semillas?
8. ¿Qué amenazas a la validez experimental identifica?

## 14. Rúbrica sugerida

| Criterio | Porcentaje |
|---|---:|
| Modelado OOP + `dataclasses` claro y correcto | 20% |
| Implementación funcional de ABC para VRP | 20% |
| Diseño experimental sobre hiperparámetros | 20% |
| Comparación con ACO y CBGA | 20% |
| Análisis crítico de resultados | 10% |
| Presentación del notebook, gráficas y conclusiones | 10% |

## 15. Extensiones opcionales

Para estudiantes avanzados:

- incorporar **penalizaciones suaves** para permitir infactibilidad temporal;
- usar una estrategia de **split** más elaborada;
- añadir **2-opt*** entre rutas;
- hibridar ABC con búsqueda local intensiva;
- incorporar una solución inicial generada por **Clarke & Wright**;
- usar valor conocido óptimo o mejor conocido para calcular **gap porcentual**.

## 16. Conclusión esperada del taller

No se espera una única respuesta universal.  
La meta es que el estudiante descubra experimentalmente que:

- **ABC** puede ser competitivo con una implementación relativamente simple;
- la calidad depende mucho del diseño del vecindario;
- el ajuste de hiperparámetros cambia radicalmente el desempeño;
- **ACO** y **CBGA** ofrecen perfiles distintos de exploración e intensificación;
- en metaheurísticas, la comparación justa requiere varias corridas y métricas claras.